In [ ]:
import hashlib
import json
import math
import re
from collections import Counter
from pathlib import Path
from statistics import fmean

from transformers import AutoTokenizer

chunk_path = Path("data/processed/chunks/sec_10k_chunks.jsonl")
proxy_path = Path(
    "data/processed/table_proxies/sec_10k_table_proxies.jsonl"
)


def load_jsonl(path: Path) -> list[dict[str, object]]:
    with path.open(encoding="utf-8") as input_file:
        return [
            json.loads(line)
            for line in input_file
            if line.strip()
        ]


chunks = load_jsonl(chunk_path)
table_chunks = [
    chunk
    for chunk in chunks
    if chunk.get("element_type") == "table"
]
proxies = load_jsonl(proxy_path)

chunk_by_id = {
    str(chunk["chunk_id"]): chunk
    for chunk in table_chunks
}

proxy_ids = [
    str(proxy["proxy_id"])
    for proxy in proxies
]
target_chunk_ids = [
    str(proxy["target_chunk_id"])
    for proxy in proxies
]

assert len(table_chunks) == 3220
assert len(proxies) == len(table_chunks)
assert len(chunk_by_id) == len(table_chunks)
assert len(set(proxy_ids)) == len(proxy_ids)
assert len(set(target_chunk_ids)) == len(target_chunk_ids)
assert set(target_chunk_ids) == set(chunk_by_id)

metadata_fields = (
    "ticker",
    "report_date",
    "table_id",
    "table_part_index",
    "table_part_count",
)

metadata_mismatches = []

for proxy in proxies:
    target_chunk_id = str(proxy["target_chunk_id"])
    source_chunk = chunk_by_id[target_chunk_id]
    proxy_text = str(proxy.get("proxy_text") or "")

    assert proxy_text.strip()
    assert str(proxy.get("ticker") or "").strip()
    assert str(proxy.get("report_date") or "").strip()

    proxy_lines = proxy_text.splitlines()

    assert "SEC item: None" not in proxy_lines
    assert "Section: None" not in proxy_lines

    expected_proxy_id = hashlib.sha256(
        f"deterministic-v1\0{target_chunk_id}".encode("utf-8")
    ).hexdigest()

    assert proxy["proxy_id"] == expected_proxy_id

    for field in metadata_fields:
        if proxy.get(field) != source_chunk.get(field):
            metadata_mismatches.append(
                {
                    "target_chunk_id": target_chunk_id,
                    "field": field,
                    "proxy_value": proxy.get(field),
                    "source_value": source_chunk.get(field),
                }
            )

assert not metadata_mismatches, metadata_mismatches[:5]

source_counts = Counter(
    str(chunk["ticker"])
    for chunk in table_chunks
)
proxy_counts = Counter(
    str(proxy["ticker"])
    for proxy in proxies
)

assert source_counts == proxy_counts

version_counts = Counter(
    str(proxy["proxy_version"])
    for proxy in proxies
)
assert version_counts == {"deterministic-v1": 3220}

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen3-Embedding-0.6B"
)

token_counts = [
    len(
        tokenizer.encode(
            str(proxy["proxy_text"]),
            add_special_tokens=False,
        )
    )
    for proxy in proxies
]

sorted_token_counts = sorted(token_counts)
p95_index = math.ceil(0.95 * len(sorted_token_counts)) - 1

missing_columns = sum(
    "\nColumns:" not in str(proxy["proxy_text"])
    for proxy in proxies
)
missing_rows = sum(
    "\nRows:" not in str(proxy["proxy_text"])
    for proxy in proxies
)
missing_both = sum(
    "\nColumns:" not in str(proxy["proxy_text"])
    and "\nRows:" not in str(proxy["proxy_text"])
    for proxy in proxies
)
context_or_header_only = sum(
    "\nRows:" not in str(proxy["proxy_text"])
    and (
        "\nContext:" in str(proxy["proxy_text"])
        or "\nColumns:" in str(proxy["proxy_text"])
    )
    for proxy in proxies
)

print("All structural checks passed!")
print(f"Table chunks: {len(table_chunks)}")
print(f"Table proxies: {len(proxies)}")
print(f"Unique proxy IDs: {len(set(proxy_ids))}")
print(f"Unique target chunk IDs: {len(set(target_chunk_ids))}")
print(f"Versions: {dict(version_counts)}")

print("\nProxy counts by ticker:")
for ticker in sorted(source_counts):
    print(
        f"{ticker}: "
        f"source={source_counts[ticker]}, "
        f"proxies={proxy_counts[ticker]}"
    )

print("\nToken statistics:")
print(f"Minimum: {min(token_counts)}")
print(f"Mean: {fmean(token_counts):.1f}")
print(f"P95: {sorted_token_counts[p95_index]}")
print(f"Maximum: {max(token_counts)}")
print(
    "Over 700 tokens: "
    f"{sum(count > 700 for count in token_counts)}"
)

print("\nMissing semantic components:")
print(f"Without Columns: {missing_columns}")
print(f"Without Rows: {missing_rows}")
print(f"Without both: {missing_both}")
print(f"Context/header only: {context_or_header_only}")

weakest_proxies = sorted(
    zip(proxies, token_counts, strict=True),
    key=lambda item: (
        sum(
            marker in str(item[0]["proxy_text"])
            for marker in (
                "\nContext:",
                "\nColumns:",
                "\nRows:",
                "\nUnits:",
            )
        ),
        item[1],
    ),
)[:5]

print("\nFive weakest proxies:")

for proxy, token_count in weakest_proxies:
    print("\n" + "=" * 80)
    print(f"ticker: {proxy['ticker']}")
    print(f"target_chunk_id: {proxy['target_chunk_id']}")
    print(f"tokens: {token_count}")
    print(proxy["proxy_text"])

IndentationError: unexpected indent (333345795.py, line 82)